In [17]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles()
styles.register_and_enable_theme()

# ---------- data ----------
NAMES = {
    "IRLTLT01JPM156N": "Japan", "IRLTLT01USM156N": "US", "IRLTLT01DEM156N": "Germany",
    "IRLTLT01GBM156N": "UK", "IRLTLT01FRM156N": "France", "IRLTLT01CAM156N": "Canada",
    "IRLTLT01ITM156N": "Italy", "IRLTLT01ESM156N": "Spain",
}
ORDER = ["Japan", "US", "Germany", "UK", "France", "Canada", "Italy", "Spain"]

wide = (
    pd.read_csv("../data/fredgraph.csv", parse_dates=["observation_date"])
    .rename(columns={"observation_date": "date", **NAMES})
    [["date"] + ORDER]
)
long = wide.melt("date", var_name="country", value_name="rate").dropna()

# ---------- hover: snap to nearest month ----------
nearest = alt.selection_point(
    fields=["date"], nearest=True, on="pointerover", clear="pointerout", empty=False
)

# ---------- encodings ----------
legend = alt.Legend(
    title=None, orient="bottom", direction="horizontal", columns=4,
    symbolType="stroke", symbolStrokeWidth=2, symbolSize=120,
    labelFontSize=10, columnPadding=16, rowPadding=2, offset=8,
)
color = alt.Color("country:N", sort=ORDER, legend=legend)
x = alt.X("date:T", title=None, axis=alt.Axis(format="%Y", tickCount=9))
y = alt.Y("rate:Q", title="%")

# ---------- layers ----------
lines = alt.Chart(long).mark_line(strokeWidth=1.8).encode(x=x, y=y, color=color)

dots = (
    alt.Chart(long).mark_circle(size=45)
    .encode(x=x, y=y, color=color, opacity=alt.condition(nearest, alt.value(1), alt.value(0)))
)

# tooltip: every country's value for the hovered month
tooltip = [alt.Tooltip("date:T", title="Month", format="%b %Y")] + [
    alt.Tooltip(f"{c}:Q", title=c, format=".2f") for c in ORDER
]
# soft 8px band (not a 1px line) so the tooltip triggers reliably anywhere you hover
hover_band = (
    alt.Chart(wide).mark_rule(color="#94a3b8", strokeWidth=8, opacity=0.25)
    .encode(x="date:T", tooltip=tooltip)
    .transform_filter(nearest)
)
catcher = alt.Chart(wide).mark_point(opacity=0).encode(x="date:T").add_params(nearest)

chart = (
    (catcher + hover_band + lines + dots)
    .properties(
        width=400, height=300,
        title=alt.Title(
            "10-year government bond yields, 2010–26",
            subtitle="Monthly average yield on 10-year government bonds, %. Source: FRED",
        ),
    )
)

styles.save(chart, path=".", name="bond_yields", svg=True)
chart

alt.LayerChart(...)

In [23]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles()
styles.register_and_enable_theme()

# ---------- data ----------
NAMES = {
    "IRLTLT01JPM156N": "Japan", "IRLTLT01USM156N": "US", "IRLTLT01DEM156N": "Germany",
    "IRLTLT01GBM156N": "UK", "IRLTLT01FRM156N": "France", "IRLTLT01CAM156N": "Canada",
    "IRLTLT01ITM156N": "Italy", "IRLTLT01ESM156N": "Spain",
}
ORDER = ["Japan", "US", "Germany", "UK", "France", "Canada", "Italy", "Spain"]

wide = (
    pd.read_csv("../data/fredgraph.csv", parse_dates=["observation_date"])
    .rename(columns={"observation_date": "date", **NAMES})
    [["date"] + ORDER]
    .query("date >= '2010-01-01'")
)
long = wide.melt("date", var_name="country", value_name="rate").dropna()

# ---------- hover: snap to nearest month ----------
nearest = alt.selection_point(
    fields=["date"], nearest=True, on="pointerover", clear="pointerout", empty=False
)

# ---------- encodings ----------
legend = alt.Legend(
    title=None, orient="bottom", direction="horizontal", columns=4,
    symbolType="stroke", symbolStrokeWidth=2, symbolSize=120,
    labelFontSize=10, columnPadding=16, rowPadding=2, offset=8,
)
color = alt.Color("country:N", sort=ORDER, legend=legend)
x = alt.X("date:T", title=None, axis=alt.Axis(format="%Y", tickCount=9))
y = alt.Y("rate:Q", title="%")

# ---------- layers ----------
lines = alt.Chart(long).mark_line(strokeWidth=1.8).encode(x=x, y=y, color=color)

dots = (
    alt.Chart(long).mark_circle(size=45)
    .encode(x=x, y=y, color=color, opacity=alt.condition(nearest, alt.value(1), alt.value(0)))
)

# tooltip: every country's value for the hovered month
tooltip = [alt.Tooltip("date:T", title="Month", format="%b %Y")] + [
    alt.Tooltip(f"{c}:Q", title=c, format=".2f") for c in ORDER
]
# soft 8px band (not a 1px line) so the tooltip triggers reliably anywhere you hover
hover_band = (
    alt.Chart(wide).mark_rule(color="#94a3b8", strokeWidth=8, opacity=0.25)
    .encode(x="date:T", tooltip=tooltip)
    .transform_filter(nearest)
)
catcher = alt.Chart(wide).mark_point(opacity=0).encode(x="date:T").add_params(nearest)

chart = (
    (catcher + hover_band + lines + dots)
    .properties(
        width=400, height=300,
       
    )
)

styles.save(chart, path=".", name="bond_yields", svg=True)
chart

alt.LayerChart(...)

In [25]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles()
styles.register_and_enable_theme()

# ---------- FRED monthly data (history) ----------
NAMES = {
    "IRLTLT01JPM156N": "Japan", "IRLTLT01USM156N": "US", "IRLTLT01DEM156N": "Germany",
    "IRLTLT01GBM156N": "UK", "IRLTLT01FRM156N": "France", "IRLTLT01CAM156N": "Canada",
    "IRLTLT01ITM156N": "Italy", "IRLTLT01ESM156N": "Spain",
}
ORDER = ["Japan", "US", "Germany", "UK", "France", "Canada", "Italy", "Spain"]

wide = (
    pd.read_csv("../data/fredgraph.csv", parse_dates=["observation_date"])
    .rename(columns={"observation_date": "date", **NAMES})
    [["date"] + ORDER]
    .query("date >= '2010-01-01'")
)

# ---------- LSEG daily data: fills Sept gaps and adds October month-to-date ----------
LSEG = {
    "Date": "date", "JP10YT=RR (BID_YIELD)": "Japan", "US10YT=RR (BID_YIELD)": "US",
    "DE10YT=RR (BID_YIELD)": "Germany", "GB10YT=RR (BID_YIELD)": "UK",
    "FR10YT=RR (BID_YIELD)": "France", "CA10YT=RR (BID_YIELD)": "Canada",
    "IT10YT=RR (BID_YIELD)": "Italy", "ES10YT=RR (BID_YIELD)": "Spain",
}
daily = (
    pd.read_excel("../data/Paul (3).xlsx", skiprows=[1])        # row 2 is just a "Close" label row
    .rename(columns=LSEG)[list(LSEG.values())]
    .assign(date=lambda d: pd.to_datetime(d["date"]))
    .set_index("date").sort_index()
)
lseg = daily.resample("MS").mean()[ORDER]                     # monthly average of daily yields
lseg = lseg.loc["2026-09-01":]                                # only the months FRED is missing/incomplete

# FRED values win where they exist; LSEG fills the gaps and adds Oct 2026
wide = wide.set_index("date").combine_first(lseg).sort_index().reset_index()
long = wide.melt("date", var_name="country", value_name="rate").dropna()

# ---------- hover: snap to nearest month ----------
nearest = alt.selection_point(
    fields=["date"], nearest=True, on="pointerover", clear="pointerout", empty=False
)

# ---------- encodings ----------
legend = alt.Legend(
    title=None, orient="bottom", direction="horizontal", columns=4,
    symbolType="stroke", symbolStrokeWidth=2, symbolSize=120,
    labelFontSize=10, columnPadding=16, rowPadding=2, offset=8,
)
color = alt.Color("country:N", sort=ORDER, legend=legend)
x = alt.X("date:T", title=None, axis=alt.Axis(format="%Y", tickCount=9))
y = alt.Y("rate:Q", title="%")

# ---------- layers ----------
lines = alt.Chart(long).mark_line(strokeWidth=1.8).encode(x=x, y=y, color=color)

dots = (
    alt.Chart(long).mark_circle(size=45)
    .encode(x=x, y=y, color=color, opacity=alt.condition(nearest, alt.value(1), alt.value(0)))
)

# tooltip: every country's value for the hovered month
tooltip = [alt.Tooltip("date:T", title="Month", format="%b %Y")] + [
    alt.Tooltip(f"{c}:Q", title=c, format=".2f") for c in ORDER
]
# soft 8px band (not a 1px line) so the tooltip triggers reliably anywhere you hover
hover_band = (
    alt.Chart(wide).mark_rule(color="#94a3b8", strokeWidth=8, opacity=0.25)
    .encode(x="date:T", tooltip=tooltip)
    .transform_filter(nearest)
)
catcher = alt.Chart(wide).mark_point(opacity=0).encode(x="date:T").add_params(nearest)

chart = (
    (catcher + hover_band + lines + dots)
    .properties(
        width=400, height=300,
       
    )
)

styles.save(chart, path=".", name="bond_yields", svg=True)
chart

alt.LayerChart(...)